In [1]:
import uproot
import numpy as np
import matplotlib.pylab as plt
import matplotlib.colors as colors
import scipy
from scipy.optimize import curve_fit # https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html
from scipy import stats
import ROOT
import zfit
import mplhep
from hepstats.splot import compute_sweights
path = '../' # set this to '' to run on the GitHub version
events_sim = uproot.open(path+'PhaseSpaceSimulation.root')
events_down = uproot.open(path+'B2HHH_MagnetDown.root')
events_up = uproot.open(path+'B2HHH_MagnetUp.root')
import os
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

/home/georg/Matter-AntimatterLab/UoM_MatterAntimatterLab/.conda/lib/python3.11/site-packages/zfit/__init__.py:93: UserWarning: TensorFlow warnings are by default suppressed by zfit. In order to show them, set the environment variable ZFIT_DISABLE_TF_WARNINGS=0. In order to suppress the TensorFlow warnings AND this warning, set ZFIT_DISABLE_TF_WARNINGS=1.
  warnings.warn(


In [2]:
def perpendicular_distance(point, start, end):
    """Compute distance from a point to a line defined by start and end."""
    line_vec = end - start
    point_vec = point - start
    line_len = np.linalg.norm(line_vec)
    if line_len == 0:
        return np.linalg.norm(point_vec)
    # Project point onto line
    proj_len = np.dot(point_vec, line_vec) / line_len
    proj_point = start + (proj_len / line_len) * line_vec
    return np.linalg.norm(point - proj_point)
def Selection_Alg_nll(prob_k, prob_pi, cut_k = None, cut_pi = None, plotting=False):
    original_probabilties = np.array(prob_k)
    probabilties = np.array(original_probabilties)
    mask = probabilties > 0
    probabilties = probabilties[mask]
   
    delta_nll = np.log(probabilties) - np.log(1-probabilties)
    cuts = np.linspace(delta_nll.min(), delta_nll.max(), 500)
    purity = []
    yield_ = []
    for c in cuts:
        mask = delta_nll > c                # select tracks above cut
        selected_K = probabilties[mask].sum()       # expected kaon yield
        n_selected = mask.sum()             # total selected tracks

        if n_selected > 0:
            purity.append(selected_K / n_selected)
        else:
            purity.append(0)
        yield_.append(selected_K)
 
    yield_ = np.array(yield_)
    purity = np.array(purity)
    cuts   = np.array(cuts)
   
    yield_output=yield_
    mask2 = yield_ > 1.0
    yield_  = yield_[mask2]
    purity = purity[mask2]
    cuts   = cuts[mask2]

    # print(yield_[0], purity[0], yield_[len(yield_)-1], purity[len(yield_)-1])
   
    distances = np.array([perpendicular_distance(np.array([y, p]),  np.array([yield_[0], purity[0]]), np.array([yield_[-1], purity[-1]])) for y, p in zip(yield_, purity)])
    knee_idx = np.argmax(distances)
    knee_cut = cuts[knee_idx]
    knee_yield = yield_[knee_idx]
    knee_purity = purity[knee_idx]
    if plotting:
        print('length',len(delta_nll))
        print(f"Suggested knee ΔNLL cut: {knee_cut:.3f}")
        print(f"Expected yield: {knee_yield:.2f}, Expected purity: {knee_purity:.2f}")
        fig,ax=plt.subplots(2,figsize=(6,5),layout='constrained',sharex=True)
        ax[0].axvline(knee_yield, color='red', linestyle='--', label=f'Cut: {knee_cut:.3f}')
        ax[0].plot(yield_, purity, marker='.', linestyle='',label='Yield-Purity line')
        ax[0].set_ylabel("Expected Sudo-Purity")
        ax[0].legend()
        gradient=(purity[-1]-purity[0])/(yield_[-1]-yield_[0])
        ax[0].plot(yield_,yield_*gradient+1,linestyle='--',color='k',label='linear fit')
        ax[1].plot(yield_,distances,label='normalised d from linear')
        ax[1].axvline(knee_yield, color='red', linestyle='--', label=f'Cut: {knee_cut:.3f}')
        ax[1].set_xlabel("Expected Kaon Sudo-Yield")
        ax[1].set_ylabel("normalised distance from linear fit")
        ax[1].legend(loc='upper left', prop={'size': 8})

        

        plt.show()
    def compute_logit_mask(p, knee_cut):
        valid = (p > 0.0) & (p < 1.0)
        logit = np.where(valid, np.log(p / (1.0 - p)), np.nan)
        mask = valid & (logit > knee_cut)
        return logit, mask
    

    logit, output_mask = compute_logit_mask(p=original_probabilties,knee_cut=knee_cut)
    # print(output_mask)
    return output_mask, knee_cut
def Selection_Alg_2D(prob_k, prob_pi, cut_k = 0.5, cut_pi = 0.5, plotting=False):
    mask = (prob_k > cut_k) & (prob_pi < cut_pi)
    return mask, (cut_k, cut_pi)
def Selection_Alg_1D_50(prob_k, prob_pi, cut_k = 0.5, cut_pi = None, plotting=False):
    return (prob_k > cut_k), cut_k
def Selection_Alg_1D_70(prob_k, prob_pi, cut_k = 0.7, cut_pi = None, plotting=False):
    return (prob_k > cut_k), cut_k

In [3]:
# taking in data
def calculating_neutral_particle_invariant_mass(data,suffix1,suffix2,m_k):
    E1=np.sqrt(data[f'{suffix1}_PX']**2+data[f'{suffix1}_PY']**2+data[f'{suffix1}_PZ']**2+m_k**2)
    E2=np.sqrt(data[f'{suffix2}_PX']**2+data[f'{suffix2}_PY']**2+data[f'{suffix2}_PZ']**2+m_k**2)
    # p=data[f'{suffix1}_PX']*data[f'{suffix2}_PX']+data[f'{suffix1}_PY']*data[f'{suffix2}_PY']+data[f'{suffix1}_PZ']*data[f'{suffix2}_PZ']
    px=data[f'{suffix1}_PX']+data[f'{suffix2}_PX']
    py=data[f'{suffix1}_PY']+data[f'{suffix2}_PY']
    pz=data[f'{suffix1}_PZ']+data[f'{suffix2}_PZ']
    return np.sqrt((E1+E2)**2-(px**2+py**2+pz**2))   
def B_meson_invariant_mass(data,m_k,mask1,mask2,mask3):
    E1=np.sqrt(data['H1_PX']**2+data['H1_PY']**2+data['H1_PZ']**2+m_k**2)
    E2=np.sqrt(data['H2_PX']**2+data['H2_PY']**2+data['H2_PZ']**2+m_k**2)
    E3=np.sqrt(data['H3_PX']**2+data['H3_PY']**2+data['H3_PZ']**2+m_k**2)
    p12=data['H1_PX']*data['H2_PX']+data['H1_PY']*data['H2_PY']+data['H1_PZ']*data['H2_PZ']
    p13=data['H1_PX']*data['H3_PX']+data['H1_PY']*data['H3_PY']+data['H1_PZ']*data['H3_PZ']
    p23=data['H2_PX']*data['H3_PX']+data['H2_PY']*data['H3_PY']+data['H2_PZ']*data['H3_PZ']
    second_term=(E1*E2)-p12+(E1*E3)-p13+(E2*E3)-p23
    m_B_squared=3*m_k**2+2*second_term
    m_B=np.sqrt(m_B_squared)[mask1 & mask2 & mask3]
    return m_B
def p_percentage_uncertainty(t):
    if t in ['x','y']:
        return np.sqrt(2)*(1*10**-3)
    else:
        return (1*10**-3)
def mass_uncertainties_B(m_k, data, m_B, mask1, mask2, mask3):

    combined_mask = mask1 & mask2 & mask3

    # --- Masked momenta ---
    px1 = data['H1_PX'][combined_mask]
    py1 = data['H1_PY'][combined_mask]
    pz1 = data['H1_PZ'][combined_mask]

    px2 = data['H2_PX'][combined_mask]
    py2 = data['H2_PY'][combined_mask]
    pz2 = data['H2_PZ'][combined_mask]

    px3 = data['H3_PX'][combined_mask]
    py3 = data['H3_PY'][combined_mask]
    pz3 = data['H3_PZ'][combined_mask]

    m_B = m_B  # already masked in B_meson_invariant_mass

    # --- Energies (masked) ---
    E1 = np.sqrt(px1**2 + py1**2 + pz1**2 + m_k**2)
    E2 = np.sqrt(px2**2 + py2**2 + pz2**2 + m_k**2)
    E3 = np.sqrt(px3**2 + py3**2 + pz3**2 + m_k**2)

    E_tot = E1 + E2 + E3

    # --- Total momentum (masked) ---
    Px_tot = px1 + px2 + px3
    Py_tot = py1 + py2 + py3
    Pz_tot = pz1 + pz2 + pz3

    # --- Momentum uncertainties ---
    eps_x = p_percentage_uncertainty('x')
    eps_y = p_percentage_uncertainty('y')
    eps_z = p_percentage_uncertainty('z')

    sigma = 0

    for px, py, pz, Ei in [(px1, py1, pz1, E1),
                           (px2, py2, pz2, E2),
                           (px3, py3, pz3, E3)]:

        sigma_px = px * eps_x
        sigma_py = py * eps_y
        sigma_pz = pz * eps_z

        dmdpx = (E_tot/Ei * px - Px_tot) / m_B
        dmdpy = (E_tot/Ei * py - Py_tot) / m_B
        dmdpz = (E_tot/Ei * pz - Pz_tot) / m_B

        sigma += (dmdpx**2) * sigma_px**2
        sigma += (dmdpy**2) * sigma_py**2
        sigma += (dmdpz**2) * sigma_pz**2

    return np.sqrt(sigma)
def neutral_mass_uncertainty(data, suffix1, suffix2, m_k, mask):

    px1 = data[f'{suffix1}_PX'][mask]
    py1 = data[f'{suffix1}_PY'][mask]
    pz1 = data[f'{suffix1}_PZ'][mask]

    px2 = data[f'{suffix2}_PX'][mask]
    py2 = data[f'{suffix2}_PY'][mask]
    pz2 = data[f'{suffix2}_PZ'][mask]

    E1 = np.sqrt(px1**2 + py1**2 + pz1**2 + m_k**2)
    E2 = np.sqrt(px2**2 + py2**2 + pz2**2 + m_k**2)

    E_tot = E1 + E2

    Px_tot = px1 + px2
    Py_tot = py1 + py2
    Pz_tot = pz1 + pz2

    m_sq = E_tot**2 - (Px_tot**2 + Py_tot**2 + Pz_tot**2)
    m = np.sqrt(m_sq)

    eps_x = p_percentage_uncertainty('x')
    eps_y = p_percentage_uncertainty('y')
    eps_z = p_percentage_uncertainty('z')

    sigma = 0

    for px, py, pz, Ei in [(px1, py1, pz1, E1),
                           (px2, py2, pz2, E2)]:

        sigma_px = px * eps_x
        sigma_py = py * eps_y
        sigma_pz = pz * eps_z

        dmdpx = (E_tot/Ei * px - Px_tot) / m
        dmdpy = (E_tot/Ei * py - Py_tot) / m
        dmdpz = (E_tot/Ei * pz - Pz_tot) / m

        sigma += (dmdpx**2) * sigma_px**2
        sigma += (dmdpy**2) * sigma_py**2
        sigma += (dmdpz**2) * sigma_pz**2

    return np.sqrt(sigma)
def D_0_filter(masses,expected_val,m_uncertanties):
    S=(masses-expected_val)/m_uncertanties
    return np.abs(S)>5
def resonance_masks(m_R_neutral_1, m_R_neutral_1_uncertainties, m_R_neutral_2, m_R_neutral_2_uncertainties, charge):
    m_R_neutral_1=np.concatenate(m_R_neutral_1)
    m_R_neutral_1_uncertainties=np.concatenate(m_R_neutral_1_uncertainties)

    m_R_neutral_2=np.concatenate(m_R_neutral_2)
    m_R_neutral_2_uncertainties=np.concatenate(m_R_neutral_2_uncertainties)


    m_high_all=np.maximum(m_R_neutral_1,m_R_neutral_2)
    m_low_all=np.minimum(m_R_neutral_1,m_R_neutral_2)

    m_high_all_uncertainties=np.maximum(m_R_neutral_1_uncertainties,m_R_neutral_2_uncertainties)
    m_low_all_uncertainties=np.minimum(m_R_neutral_1_uncertainties,m_R_neutral_2_uncertainties)

    charge=np.concatenate(charge)


    D_0_mask1=D_0_filter(m_high_all,1864.84,m_high_all_uncertainties)

    D_0_mask2=D_0_filter(m_low_all,1864.84,m_low_all_uncertainties)


    chi_meson_removal_mask_1 = (~((m_high_all > 3404.21) & (m_high_all < 3425.21)))

    chi_meson_removal_mask_2 = (~((m_low_all > 3404.21) & (m_low_all < 3425.21)))
    total_removal_mask=(D_0_mask1 & chi_meson_removal_mask_1) & (D_0_mask2 & chi_meson_removal_mask_2)

    return total_removal_mask,charge
def process_tree(tree, Selection_Alg):
    pT = []
    pX = []
    pY = []
    pZ = []
    prob_k=[]
    prob_pi=[]
    m_B_all=[]
    m_B_all_uncertainties=[]
    charge=[]


    m_R_neutral_1=[]
    m_R_neutral_1_uncertainties=[]
    m_R_neutral_2=[]
    m_R_neutral_2_uncertainties=[]
    m_high_all=[]
    m_low_all=[]
    event_counter = 0
    for tree in trees:
        # This outer loop is a technical loop of uproot over chunks of events
    #    for data in tree.iterate(['H*_P[XYZ]','H*_Charge','H*_Prob*','H*_isMuon'], library='np'): #not working ...
        for data in tree.iterate( library='np'):
            # As Python can handle calculations with arrays, we can calculate derived quantities here
            pT_H1 = np.sqrt(data['H1_PX']**2+data['H1_PY']**2)
            pT_H2 = np.sqrt(data['H2_PX']**2+data['H2_PY']**2)
            pT_H3 = np.sqrt(data['H3_PX']**2+data['H3_PY']**2)
            

            # B_meson invariant mass


            m_k=493.667
            # print(len(data['H2_ProbK']))

            # mask1=(Selection_Alg(original_probabilties=data['H1_ProbK']/(data['H1_ProbK']+data['H1_ProbPi']),plotting=False)[0])
            # mask2=(Selection_Alg(original_probabilties=data['H2_ProbK']/(data['H2_ProbK']+data['H2_ProbPi']),plotting=False)[0])
            # mask3=(Selection_Alg(original_probabilties=data['H3_ProbK']/(data['H3_ProbK']+data['H3_ProbPi']),plotting=False)[0])

            with ThreadPoolExecutor(max_workers=3) as executor:
                f1 = executor.submit(Selection_Alg, data['H1_ProbK'], data['H1_ProbPi'])
                f2 = executor.submit(Selection_Alg, data['H2_ProbK'], data['H2_ProbPi'])
                f3 = executor.submit(Selection_Alg, data['H3_ProbK'], data['H3_ProbPi'])
                mask1 = f1.result()[0]
                mask2 = f2.result()[0]
                mask3 = f3.result()[0]


            # mask1=np.full(len(data['H1_ProbK']),True)
            # mask2=np.full(len(data['H2_ProbK']),True)
            # mask3=np.full(len(data['H2_ProbK']),True)

            # mask1=data['H1_ProbK']>0.9
            # mask2=data['H2_ProbK']>0.9
            # mask3=data['H2_ProbK']>0.9
            total_mask=mask1 & mask2 & mask3

            m_B=B_meson_invariant_mass(data,m_k,mask1,mask2,mask3)
            m_B_uncertainty=mass_uncertainties_B(m_k,data,m_B,mask1,mask2,mask3)
            m_B_all.append(m_B)
            m_B_all_uncertainties.append(m_B_uncertainty)
            
            

            # neutral particles invariant mass
            # data_neutral_mask=(data['H1_Charge'][mask1 & mask2 & mask3] + data['H2_Charge'][mask1 & mask2 & mask3] + data['H3_Charge'][mask1 & mask2 & mask3]).abs()==1
            data_neutral_mask=np.abs(data['H1_Charge'] + data['H2_Charge'] + data['H3_Charge'])==1
            event_mask=data_neutral_mask & mask1 & mask2 & mask3
            H1_different_mask=event_mask & (data['H1_Charge']!=data['H2_Charge']) & (data['H1_Charge']!=data['H3_Charge'])
            H2_different_mask=event_mask & (data['H2_Charge']!=data['H1_Charge']) & (data['H2_Charge']!=data['H3_Charge'])
            H3_different_mask=event_mask & (data['H3_Charge']!=data['H2_Charge']) & (data['H3_Charge']!=data['H1_Charge'])

            m12=calculating_neutral_particle_invariant_mass(data,'H1','H2',m_k)
            m13=calculating_neutral_particle_invariant_mass(data,'H1','H3',m_k)
            m23=calculating_neutral_particle_invariant_mass(data,'H2','H3',m_k)
            
            sigma12 = neutral_mass_uncertainty(data, 'H1', 'H2', m_k, event_mask)
            sigma13 = neutral_mass_uncertainty(data, 'H1', 'H3', m_k, event_mask)
            sigma23 = neutral_mass_uncertainty(data, 'H2', 'H3', m_k, event_mask)

            m_R_neutral_1.append((H1_different_mask*m12 + H2_different_mask*m12 + H3_different_mask*m13)[event_mask])
            m_R_neutral_2.append((H1_different_mask*m13 + H2_different_mask*m23 + H3_different_mask*m23)[event_mask])

            m_R_neutral_1_uncertainties.append(
            np.where(H1_different_mask[event_mask], sigma12,
                np.where(H2_different_mask[event_mask], sigma12,
                        np.where(H3_different_mask[event_mask], sigma13, np.nan))))

            m_R_neutral_2_uncertainties.append(
            np.where(H1_different_mask[event_mask], sigma13,
                np.where(H2_different_mask[event_mask], sigma23,
                        np.where(H3_different_mask[event_mask], sigma23, np.nan))))
            

            event_charge=data['H1_Charge']+data['H2_Charge']+data['H3_Charge']
            charge.append(event_charge[mask1 & mask2 & mask3])

            # This loop will go over individual events
            # Vectorised masks replacing the per-event if-checks
            pz_mask = (data['H1_PZ'] > 0) & (data['H2_PZ'] > 0) & (data['H3_PZ'] > 0)
            muon_mask = (data['H1_isMuon'] == 0) & (data['H2_isMuon'] == 0) & (data['H3_isMuon'] == 0)
            event_mask = pz_mask & muon_mask

            # MAX_EVENTS limit


            event_counter += event_mask.sum()
            print('Read', event_counter, 'events')

            # Fill arrays vectorised
            pT.append(np.concatenate([pT_H1[event_mask], pT_H2[event_mask], pT_H3[event_mask]]))
            pX.append(np.concatenate([data['H1_PX'][event_mask], data['H2_PX'][event_mask], data['H3_PX'][event_mask]]))
            pY.append(np.concatenate([data['H1_PY'][event_mask], data['H2_PY'][event_mask], data['H3_PY'][event_mask]]))
            pZ.append(np.concatenate([data['H1_PZ'][event_mask], data['H2_PZ'][event_mask], data['H3_PZ'][event_mask]]))

            prob_k.append(np.concatenate([data['H1_ProbK'][event_mask], data['H2_ProbK'][event_mask], data['H3_ProbK'][event_mask]]))
            prob_pi.append(np.concatenate([data['H1_ProbPi'][event_mask], data['H2_ProbPi'][event_mask], data['H3_ProbPi'][event_mask]]))

    prob_k = np.concatenate(prob_k)
    prob_pi = np.concatenate(prob_pi)
    pT = np.concatenate(pT)
    pX = np.concatenate(pX)
    pY = np.concatenate(pY)
    pZ = np.concatenate(pZ)

    m_B_all=np.concatenate(m_B_all)
    m_B_all_uncertainties=np.concatenate(m_B_all_uncertainties)

    total_removal_mask, charge = resonance_masks(m_R_neutral_1, m_R_neutral_1_uncertainties, m_R_neutral_2, m_R_neutral_2_uncertainties, charge)


    m_B_all=m_B_all[total_removal_mask]
    charge=charge[total_removal_mask]

    positive_B_mask=(charge==1)
    negative_B_mask=(charge==-1)
    return m_B_all, positive_B_mask, negative_B_mask
# Unpacking Tree Data
# Note that the simulation tree is called 'PhaseSpaceTree' and does not have the ProbPi/K variables filled.
print('Input data variables:')
print(events_up['DecayTree'].keys())

# Select which set of input data is to be analysed. Uncomment exactly one line
# trees = [events_sim['PhaseSpaceTree']]                       # Simulation
# trees = [events_down['DecayTree']]                          # Magnet down data
#trees = [events_up['DecayTree']]                             # Magnet up data
trees = [events_down['DecayTree'],events_up['DecayTree']]   # Magnet down+up data
executor = ThreadPoolExecutor(max_workers=os.cpu_count()-1)
# This loop goes over the trees to be analysed
m_B_all = []
positive_B_mask = []
negative_B_mask = []
#selection_algorithms = [Selection_Alg_2D, Selection_Alg_1D_50, Selection_Alg_1D_70]
selection_algorithms = [Selection_Alg_nll, Selection_Alg_2D, Selection_Alg_1D_50, Selection_Alg_1D_70]
columns = ['nll', '2D', '1D_50', '1D_70']
df = pd.concat([
    pd.DataFrame({
        "m_B": m_B,
        "positive_mask": pos,
        "negative_mask": neg,
        "algorithm": col
    })
    for (alg, col) in zip(selection_algorithms, columns)
    for (m_B, pos, neg) in [process_tree(trees, alg)]
], ignore_index=True)




Input data variables:
['B_FlightDistance', 'B_VertexChi2', 'H1_PX', 'H1_PY', 'H1_PZ', 'H1_ProbK', 'H1_ProbPi', 'H1_Charge', 'H1_IPChi2', 'H1_isMuon', 'H2_PX', 'H2_PY', 'H2_PZ', 'H2_ProbK', 'H2_ProbPi', 'H2_Charge', 'H2_IPChi2', 'H2_isMuon', 'H3_PX', 'H3_PY', 'H3_PZ', 'H3_ProbK', 'H3_ProbPi', 'H3_Charge', 'H3_IPChi2', 'H3_isMuon']


/tmp/ipykernel_14373/1381424241.py:72: RuntimeWarning: invalid value encountered in log
  logit = np.where(valid, np.log(p / (1.0 - p)), np.nan)


Read 576320 events
Read 1153098 events
Read 1729272 events
Read 2304688 events
Read 2880510 events
Read 3456748 events
Read 3806297 events
Read 4401243 events
Read 4995899 events
Read 5590141 events
Read 6185255 events
Read 6311517 events
Read 576320 events
Read 1153098 events
Read 1729272 events
Read 2304688 events
Read 2880510 events
Read 3456748 events
Read 3806297 events
Read 4401243 events
Read 4995899 events
Read 5590141 events
Read 6185255 events
Read 6311517 events
Read 576320 events
Read 1153098 events
Read 1729272 events
Read 2304688 events
Read 2880510 events
Read 3456748 events
Read 3806297 events
Read 4401243 events
Read 4995899 events
Read 5590141 events
Read 6185255 events
Read 6311517 events
Read 576320 events
Read 1153098 events
Read 1729272 events
Read 2304688 events
Read 2880510 events
Read 3456748 events
Read 3806297 events
Read 4401243 events
Read 4995899 events
Read 5590141 events
Read 6185255 events
Read 6311517 events


### 2 Gaussian Fits + exponential

In [11]:
def root_fitting(m_B_all, mask, canvas, title="B-mass fit"):
    # Make canvas current
    canvas.cd()
    canvas.Clear()

    # RooRealVar
    mB = ROOT.RooRealVar("mB", "B mass [MeV]", 5050, 6300)



    # Signal Gaussian 1
    mean1 = ROOT.RooRealVar("mean1", "signal mean", 5279.31, 5260, 5290) #5290)
    sigma1 = ROOT.RooRealVar("sigma1", "signal sigma", 17, 1, 50)
    N_s1 = ROOT.RooRealVar("N_s1", "signal yield", 10000, 0, 50000)
    gauss1 = ROOT.RooGaussian("gauss1", "signal gaussian", mB, mean1, sigma1)

    # Background exponential
    N_bkg = ROOT.RooRealVar("N_bkg", "bkg yield", 200, 0, 5000)
    k = ROOT.RooRealVar("k", "exp slope", 0.006, -0.1, 0.1)
    exp_bkg = ROOT.RooExponential("exp_bkg", "bkg exp", mB, k)

    # Cut Gaussian
    mean2 = ROOT.RooRealVar("mean2", "cut gaussian mean", 5075, 5050, 5100)
    sigma2 = ROOT.RooRealVar("sigma2", "cut gaussian sigma", 50, 1, 100)
    N2 = ROOT.RooRealVar("N2", "cut gaussian yield", 1000, 0, 5000)
    gauss2 = ROOT.RooGaussian("gauss2", "cut gaussian", mB, mean2, sigma2)

    # Extended PDFs
    ext_gauss1 = ROOT.RooExtendPdf("ext_gauss1", "ext_gauss1", gauss1, N_s1)
    ext_gauss2 = ROOT.RooExtendPdf("ext_gauss2", "ext_gauss2", gauss2, N2)
    ext_expo = ROOT.RooExtendPdf("ext_expo", "ext_expo", exp_bkg, N_bkg)

    # Combined model
    model = ROOT.RooAddPdf("model", "Signal + Cut + Bkg", ROOT.RooArgList(ext_gauss1, ext_gauss2, ext_expo))

    # Create RooDataSet
    data = ROOT.RooDataSet("data", "unbinned data", ROOT.RooArgSet(mB))

    for val in m_B_all[mask]:
        mB.setVal(val)
        data.add(ROOT.RooArgSet(mB))
    result = model.fitTo(data, ROOT.RooFit.Extended(True), ROOT.RooFit.Save())

    # # Print fit results
    # print(f"Fit results ({title}):")
    # for var in [mean1, sigma1, N_s1, mean2, sigma2, N2, k, N_bkg]:
    #     print(f"{var.GetName():6s}: {var.getVal():.2f} ± {var.getError():.2f}")

    # Plot
    frame = mB.frame(ROOT.RooFit.Title(title))
    data.plotOn(frame,ROOT.RooFit.Name("data_hist"),ROOT.RooFit.Binning(250))
    model.plotOn(frame, ROOT.RooFit.LineColor(ROOT.kRed), ROOT.RooFit.Name("Total"))
    model.plotOn(frame, ROOT.RooFit.Components("ext_gauss1"), ROOT.RooFit.LineColor(ROOT.kGreen), ROOT.RooFit.Name("Signal"))
    model.plotOn(frame, ROOT.RooFit.Components("ext_gauss2"), ROOT.RooFit.LineColor(ROOT.kPink), ROOT.RooFit.Name("CutGaussian"))
    model.plotOn(frame, ROOT.RooFit.Components("ext_expo"), ROOT.RooFit.LineColor(ROOT.kBlue), ROOT.RooFit.Name("Background"))

    frame.BuildLegend()
    frame.Draw()
    canvas.Update()
    
    n_params = result.floatParsFinal().getSize()
    chi2_red = frame.chiSquare("Total","data_hist",n_params)

    # print("Reduced chi2:", chi2_red)
    # print("Chi2:", chi2_red*182)

        # Open a new ROOT file
    f = ROOT.TFile(f"fit_results{title}.root", "RECREATE")

    # Save dataset
    data.Write("data")

    # Save model
    model.Write("model")

    # Save yield variables
    result.floatParsFinal().find("N_s1").Write("N_s1")
    result.floatParsFinal().find("N2").Write("N2")
    result.floatParsFinal().find("N_bkg").Write("N_bkg")

    f.Close()


    # ----------------------------
    # Return useful objects
    # ----------------------------
    return {
        "result": result,
        "model": model,
        "frame": frame,
        "chi2_red": chi2_red,
        "N_s1": N_s1,
        "N2": N2,
        "N_bkg": N_bkg,
        "data": data
    }



def chi_squared_poisson(fit,bin_centres,sigma):

    return np.sum((bin_centres-fit)/sigma**2)




### 2 Cauchy and exponential Refitting

In [12]:
def root_fitting_cauchy(m_B_all,mask,canvas,title="B-mass fit"):

    canvas.cd()
    canvas.Clear()

    # RooRealVar
    mB = ROOT.RooRealVar("mB", "B mass [MeV]", 5050, 6300)



    # Signal cauchy 1
    mean1 = ROOT.RooRealVar("mean1", "signal mean", 5279.31, 5260, 5290) #5290)
    sigma1 = ROOT.RooRealVar("sigma1", "signal sigma", 7, 1, 50)
    N_s1 = ROOT.RooRealVar("N_s1", "signal yield", 10000, 0, 50000)
    # gauss1 = ROOT.Roocauchy("gauss1", "signal cauchy", mB, mean1, sigma1)
    cauchy1 = ROOT.RooBreitWigner("cauchy1","signal cauchy",mB,mean1,sigma1) #using sigma as HWHH

    # Background exponential
    N_bkg = ROOT.RooRealVar("N_bkg", "bkg yield", 200, 0, 5000)
    k = ROOT.RooRealVar("k", "exp slope", 0.006, -0.1, 0.1)
    exp_bkg = ROOT.RooExponential("exp_bkg", "bkg exp", mB,k)

    # Cut cauchy
    mean2 = ROOT.RooRealVar("mean2", "cut cauchy mean", 5075, 5050, 5100)
    sigma2 = ROOT.RooRealVar("sigma2", "cut cauchy sigma", 15, 1, 100)
    N2 = ROOT.RooRealVar("N2", "cut cauchy yield", 1000, 0, 5000)
    cauchy2 = ROOT.RooBreitWigner("cauchy2", "cut cauchy", mB, mean2, sigma2)

    # Extended PDFs
    ext_gauss1 = ROOT.RooExtendPdf("ext_cauchy1", "ext_cauchy1", cauchy1, N_s1)
    ext_gauss2 = ROOT.RooExtendPdf("ext_cauchy2", "ext_cauchy2", cauchy2, N2)
    ext_expo = ROOT.RooExtendPdf("ext_uniform", "ext_uniform", exp_bkg, N_bkg)

    # Combined model
    model = ROOT.RooAddPdf("model", "Signal + Cut + Bkg", ROOT.RooArgList(ext_gauss1, ext_gauss2, ext_expo))

    # Create RooDataSet
    data = ROOT.RooDataSet("data", "unbinned data", ROOT.RooArgSet(mB))

    for val in m_B_all[mask]:
        mB.setVal(val)
        data.add(ROOT.RooArgSet(mB))
    result = model.fitTo(data, ROOT.RooFit.Extended(True), ROOT.RooFit.Save())

    # # Print fit results
    # print(f"Fit results ({title}):")
    # for var in [mean1, sigma1, N_s1, mean2, sigma2, N2, k, N_bkg]:
    #     print(f"{var.GetName():6s}: {var.getVal():.2f} ± {var.getError():.2f}")

    # Plot
    frame = mB.frame(ROOT.RooFit.Title(title))
    data.plotOn(frame,ROOT.RooFit.Name("data_hist"),ROOT.RooFit.Binning(250))
    model.plotOn(frame, ROOT.RooFit.LineColor(ROOT.kRed), ROOT.RooFit.Name("Total"))
    model.plotOn(frame, ROOT.RooFit.Components("ext_gauss1"), ROOT.RooFit.LineColor(ROOT.kGreen), ROOT.RooFit.Name("Signal"))
    model.plotOn(frame, ROOT.RooFit.Components("ext_gauss2"), ROOT.RooFit.LineColor(ROOT.kPink), ROOT.RooFit.Name("Cutcauchy"))
    model.plotOn(frame, ROOT.RooFit.Components("ext_expo"), ROOT.RooFit.LineColor(ROOT.kBlue), ROOT.RooFit.Name("Background"))

    frame.BuildLegend()
    frame.Draw()
    canvas.Update()
    
    n_params = result.floatParsFinal().getSize()
    chi2_red = frame.chiSquare("Total","data_hist",n_params)

    # print("Reduced chi2:", chi2_red)
    # print("Chi2:", chi2_red*182)

        # Open a new ROOT file
    f = ROOT.TFile(f"fit_results{title}.root", "RECREATE")

    # Save dataset
    data.Write("data")

    # Save model
    model.Write("model")

    # Save yield variables
    result.floatParsFinal().find("N_s1").Write("N_s1")
    result.floatParsFinal().find("N2").Write("N2")
    result.floatParsFinal().find("N_bkg").Write("N_bkg")

    f.Close()


    # ----------------------------
    # Return useful objects
    # ----------------------------
    return {
        "result": result,
        "model": model,
        "frame": frame,
        "chi2_red": chi2_red,
        "N_s1": N_s1,
        "N2": N2,
        "N_bkg": N_bkg,
        "data": data
    }


### 2 voigtian and exponential

In [13]:
def root_fitting_voigtian(m_B_all, mask, canvas, title="B-mass fit"):
    canvas.cd()
    canvas.Clear()

    # ------------------------
    # RooRealVar: mass variable
    # ------------------------
    mB = ROOT.RooRealVar("mB", "B mass [MeV]", 5050, 6300)

    # ------------------------
    # Signal Voigtian 1
    # ------------------------
    mean1  = ROOT.RooRealVar("mean1", "signal mean", 5279.31, 5260, 5290)
    sigma1 = ROOT.RooRealVar("sigma1", "detector resolution", 5, 0.1, 50)  # detector resolution
    gamma1 = ROOT.RooRealVar("gamma1","natural width", 0.005, 0, 1)       # natural width
    N_s1   = ROOT.RooRealVar("N_s1", "signal yield", 10000, 0, 50000)
    voigt1 = ROOT.RooVoigtian("voigt1", "signal Voigtian", mB, mean1, sigma1, gamma1)

    # ------------------------
    # Cut Voigtian 2
    # ------------------------
    mean2  = ROOT.RooRealVar("mean2", "cut mean", 5075, 5050, 5100)
    sigma2 = ROOT.RooRealVar("sigma2", "cut sigma", 5, 0.1, 50)
    gamma2 = ROOT.RooRealVar("gamma2","cut gamma", 0.005, 0, 1)
    N2     = ROOT.RooRealVar("N2", "cut yield", 1000, 0, 5000)
    voigt2 = ROOT.RooVoigtian("voigt2", "cut Voigtian", mB, mean2, sigma2, gamma2)

    # ------------------------
    # Background exponential
    # ------------------------
    k      = ROOT.RooRealVar("k", "exp slope", 0.006, -0.1, 0.1)
    N_bkg  = ROOT.RooRealVar("N_bkg", "bkg yield", 200, 0, 5000)
    exp_bkg = ROOT.RooExponential("exp_bkg", "background exp", mB, k)

    # ------------------------
    # Combined model with yields
    # ------------------------
    model = ROOT.RooAddPdf(
        "model",
        "Signal + Cut + Bkg",
        ROOT.RooArgList(voigt1, voigt2, exp_bkg),
        ROOT.RooArgList(N_s1, N2, N_bkg)
    )

    # ------------------------
    # Create RooDataSet
    # ------------------------
    data = ROOT.RooDataSet("data", "unbinned data", ROOT.RooArgSet(mB))
    for val in m_B_all[mask]:
        mB.setVal(val)
        data.add(ROOT.RooArgSet(mB))

    # ------------------------
    # Fit
    # ------------------------
    result = model.fitTo(data, ROOT.RooFit.Extended(True), ROOT.RooFit.Save())

    # Print fit results
    # print(f"Fit results ({title}):")
    # for var in [mean1, sigma1, gamma1, N_s1, mean2, sigma2, gamma2, N2, k, N_bkg]:
    #     print(f"{var.GetName():6s}: {var.getVal():.2f} ± {var.getError():.2f}")

    # ------------------------
    # Plot
    # ------------------------
    frame = mB.frame(ROOT.RooFit.Title(title))
    data.plotOn(frame, ROOT.RooFit.Binning(250), ROOT.RooFit.Name("data_hist"))

    # Total model
    model.plotOn(frame, ROOT.RooFit.LineColor(ROOT.kRed), ROOT.RooFit.Name("Total"))

    # Individual components
    model.plotOn(frame, ROOT.RooFit.Components("voigt1"), ROOT.RooFit.LineColor(ROOT.kGreen), ROOT.RooFit.Name("Signal"))
    model.plotOn(frame, ROOT.RooFit.Components("voigt2"), ROOT.RooFit.LineColor(ROOT.kPink), ROOT.RooFit.Name("CutVoigt"))
    model.plotOn(frame, ROOT.RooFit.Components("exp_bkg"), ROOT.RooFit.LineColor(ROOT.kBlue), ROOT.RooFit.Name("Background"))

    frame.BuildLegend()
    frame.Draw()
    canvas.Update()

    # ------------------------
    # Chi2
    # ------------------------
    n_params = result.floatParsFinal().getSize()
    chi2_red = frame.chiSquare("Total", "data_hist", n_params)
    # print("Reduced chi2:", chi2_red)
    # print("Chi2:", chi2_red*len(m_B_all[mask]))

    # ------------------------
    # Save results
    # ------------------------
    f = ROOT.TFile(f"fit_results_{title}.root", "RECREATE")
    data.Write("data")
    model.Write("model")
    for var in [N_s1, N2, N_bkg]:
        result.floatParsFinal().find(var.GetName()).Write(var.GetName())
    f.Close()

    # ------------------------
    # Return useful objects
    # ------------------------
    return {
        "result": result,
        "model": model,
        "frame": frame,
        "chi2_red": chi2_red,
        "N_s1": N_s1,
        "N2": N2,
        "N_bkg": N_bkg,
        "data": data
    }

### Assymetry and Dalitz Plots

In [14]:
def Asymmetry(N_pos,N_neg):
    return (N_neg-N_pos)/(N_neg+N_pos)
def Asymmetry_uncertainty_with_N(N_pos,N_pos_err,N_neg,N_neg_err):
    # return np.sqrt((4/(N_pos+N_neg)**4)*(N_neg**2*N_pos_err**2+N_pos**2*N_neg_err**2))
    return ((N_neg_err/(N_neg+N_pos)**4)**2)+(N_pos_err/(N_neg+N_pos)**4)**2
def Asymmetry_uncertainty_with_A(N_pos,N_neg,A):
    return np.sqrt((1-A**2)/(N_neg+N_pos))
fitting_models = [root_fitting, root_fitting_cauchy, root_fitting_voigtian]
fitting_models = [root_fitting, root_fitting_cauchy, root_fitting_voigtian]
final_results = {}
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.ERROR)
ROOT.gROOT.ProcessLine("gErrorIgnoreLevel = kError;")
for fit_func in fitting_models:
    final_results[fit_func.__name__] = {}
    for alg, group in df.groupby("algorithm"):
        final_results[fit_func.__name__][alg] = {}
        c1 = ROOT.TCanvas(f"c1_{fit_func.__name__}_{alg}", "Positive B", 800, 600)
        c2 = ROOT.TCanvas(f"c2_{fit_func.__name__}_{alg}", "Negative B", 800, 600)
        result_pos_unbinned = (fit_func(group['m_B'], group['positive_mask'], c1, "Bpos_mass_fit"))
        result_neg_unbinned = (fit_func(group['m_B'], group['negative_mask'], c2, "Bneg_mass_fit"))
        c1.SaveAs(f"fit_plots/Bpos_{fit_func.__name__}_{alg}.png")
        c2.SaveAs(f"fit_plots/Bneg_{fit_func.__name__}_{alg}.png")

        A = final_results[fit_func.__name__][alg]['A'] = Asymmetry(result_pos_unbinned['result'].floatParsFinal().find("N_s1").getVal(),
         result_neg_unbinned['result'].floatParsFinal().find("N_s1").getVal())

        A_uncertainty_with_A = float(Asymmetry_uncertainty_with_A(result_pos_unbinned['result'].floatParsFinal().find("N_s1").getVal(), 
        result_neg_unbinned['result'].floatParsFinal().find("N_s1").getVal(), final_results[fit_func.__name__][alg]['A']))

        final_results[fit_func.__name__][alg]['A'] = A
        final_results[fit_func.__name__][alg]['A_uncertainty_with_A'] = A_uncertainty_with_A

# A = [
#     Asymmetry(
#         pos['result'].floatParsFinal().find("N_s1").getVal(),
#         neg['result'].floatParsFinal().find("N_s1").getVal()
#     )
#     for pos, neg in zip(result_pos_unbinned, result_neg_unbinned)
# ]
# A_uncertainty_with_A = [float(
#     Asymmetry_uncertainty_with_A(
#         pos['result'].floatParsFinal().find("N_s1").getVal(),
#         neg['result'].floatParsFinal().find("N_s1").getVal(),
#         A[i]
#     ))
#     for i, (pos, neg) in enumerate(zip(result_pos_unbinned, result_neg_unbinned))
# ]
print(final_results)

{'root_fitting': {'1D_50': {'A': -0.052806178684444774, 'A_uncertainty_with_A': 0.0077201743425679295}, '1D_70': {'A': -0.06020379461148076, 'A_uncertainty_with_A': 0.011557492917929553}, '2D': {'A': -0.05161273675884262, 'A_uncertainty_with_A': 0.007998046144829163}, 'nll': {'A': -0.05343159472427146, 'A_uncertainty_with_A': 0.007132540132833311}}, 'root_fitting_cauchy': {'1D_50': {'A': -0.05517004856612665, 'A_uncertainty_with_A': 0.007217405497327685}, '1D_70': {'A': -0.06120532484022774, 'A_uncertainty_with_A': 0.010957603026545654}, '2D': {'A': -0.055510885088949566, 'A_uncertainty_with_A': 0.007507604885097036}, 'nll': {'A': -0.05976630767536488, 'A_uncertainty_with_A': 0.006673200772122973}}, 'root_fitting_voigtian': {'1D_50': {'A': -0.05564407340677167, 'A_uncertainty_with_A': 0.007202150400603835}, '1D_70': {'A': -0.06107051825210786, 'A_uncertainty_with_A': 0.01092834349908001}, '2D': {'A': -0.0560511044476879, 'A_uncertainty_with_A': 0.007490555914745242}, 'nll': {'A': -0.06

In [19]:
from IPython.display import display, HTML

rows = []
for fit_func, alg_results in final_results.items():
    for alg, values in alg_results.items():
        rows.append({
            "Fit Function": fit_func,
            "Algorithm": alg,
            "A ± σ": f"{values['A']:.4f} ± {values['A_uncertainty_with_A']:.4f}"
        })

df_results = pd.DataFrame(rows)
display(HTML(df_results.to_html(index=False)))

Fit Function,Algorithm,A ± σ
root_fitting,1D_50,-0.0528 ± 0.0077
root_fitting,1D_70,-0.0602 ± 0.0116
root_fitting,2D,-0.0516 ± 0.0080
root_fitting,nll,-0.0534 ± 0.0071
root_fitting_cauchy,1D_50,-0.0552 ± 0.0072
root_fitting_cauchy,1D_70,-0.0612 ± 0.0110
root_fitting_cauchy,2D,-0.0555 ± 0.0075
root_fitting_cauchy,nll,-0.0598 ± 0.0067
root_fitting_voigtian,1D_50,-0.0556 ± 0.0072
root_fitting_voigtian,1D_70,-0.0611 ± 0.0109
